# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 6: Launch Site Locations Analysis with Folium

Launch success may depend on where a rocket is launched from. This notebook maps the launch sites, the outcome of every launch, and the distance from a launch site to nearby features.

**Objectives**
- Mark all launch sites on a map
- Mark the successful and failed launches for each site
- Calculate the distances between a launch site and its proximities

In [1]:
%pip install -q folium

In [2]:
import math
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

In [3]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

# One row per launch site with its coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### Task 1: Mark all launch sites on a map

In [4]:
# Start the map at NASA Johnson Space Center, Houston
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % 'NASA JSC'))
site_map.add_child(circle)
site_map.add_child(marker)

# Add a circle and a text label for each launch site
for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(site['Launch Site'])).add_to(site_map)
    folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site['Launch Site'])
    ).add_to(site_map)

site_map

### Task 2: Mark the successful and failed launches for each site
Green markers are successful landings (`class = 1`) and red markers are failures (`class = 0`). Launches at the same site share coordinates, so they are grouped in marker clusters. Click a cluster to expand it.

In [5]:
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [6]:
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for _, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=record['Launch Site'] + (' : success' if record['class'] == 1 else ' : failure'))
    marker_cluster.add_child(marker)

site_map

In [7]:
# Success rate per launch site, to read alongside the map
spacex_df.groupby('Launch Site')['class'].agg(launches='count', successes='sum', success_rate='mean').round(3)

,launches,successes,success_rate
Launch Site,,,
CCAFS LC-40,26,7,0.269
CCAFS SLC-40,7,3,0.429
KSC LC-39A,13,10,0.769
VAFB SLC-4E,10,4,0.400


### Task 3: Calculate the distances between a launch site and its proximities
`MousePosition` shows the coordinates under the mouse pointer, which is how the proximity points below were read from the map.

In [8]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)
site_map

In [9]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Distance in km between two points, using the haversine formula."""
    R = 6373.0  # approximate radius of the Earth in km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

In [10]:
# Launch site used for the proximity analysis
site_name = 'CCAFS SLC-40'
match = launch_sites_df[launch_sites_df['Launch Site'] == site_name]
if match.empty:
    match = launch_sites_df.iloc[[0]]
    site_name = match.iloc[0]['Launch Site']
launch_site_lat = float(match.iloc[0]['Lat'])
launch_site_lon = float(match.iloc[0]['Long'])
print(site_name, launch_site_lat, launch_site_lon)

# Closest coastline point, read from the map with MousePosition
coastline_lat, coastline_lon = 28.56367, -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print("Distance to coastline: {:.2f} km".format(distance_coastline))

CCAFS SLC-40 28.56319718 -80.57682003
Distance to coastline: 0.51 km


In [11]:
# Marker showing the distance, and a line from the launch site to the coastline
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline)))
site_map.add_child(distance_marker)

lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=1)
site_map.add_child(lines)
site_map

The same is done for the closest railway, highway and city. Zoom in on the launch site to see the lines.

In [12]:
proximities = {
    'Railway': (28.57206, -80.58525),
    'Highway': (28.56327, -80.57076),
    'City (Titusville)': (28.61200, -80.80788),
}

distances = {'Coastline': distance_coastline}
for name, (lat, lon) in proximities.items():
    distance = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    distances[name] = distance
    folium.Marker(
        [lat, lon],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % "{} {:.2f} KM".format(name, distance))
    ).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=1).add_to(site_map)

pd.DataFrame({'Proximity': list(distances.keys()),
              'Distance from ' + site_name + ' (km)': [round(d, 2) for d in distances.values()]})

,Proximity,Distance from CCAFS SLC-40 (km)
0,Coastline,0.51
1,Railway,1.28
2,Highway,0.59
3,City (Titusville),23.21


In [13]:
# Zoomed-in view of the launch site and its proximities
site_map.location = [launch_site_lat, launch_site_lon]
site_map.options['zoom'] = 12
site_map